# Llm deliberation bootstrap

This notebook belongs to the project's sequential measurement and validation programme. Read its result as evidence about behavioural validity, representation, comparator strength, timing, information matching, or mechanistic calibration as appropriate. Legacy H1/H_priv identifiers may remain inside code, saved paths, or frozen schemas for reproducibility; they are not the object being "found" by the current analysis.

**Repository framing.** The current paper separates target-specific representation from predictive privilege. A positive neural result is interpreted only after behavioural validity, control, comparator-strength, timing, and information-set checks.


# Latent reservations — Notebook 00
## LLM-Deliberation zero-to-pilot bootstrap and verification

This is the **first execution notebook** for *Latent reservations*. It is designed for a fresh GPU Runpod and now defaults to doing the work rather than dry-running it:

1. install the Python dependencies into the active Jupyter kernel environment;
2. record the Runpod/GPU/software environment;
3. clone and pin the official LLM-Deliberation repository;
4. create a project-owned copy of the upstream base game configured to use the local subject model;
5. run the authors' native 24-round simulation path;
6. load/download the 7–8B subject checkpoint and verify a **pre-first-token residual-stream hook**;
7. define the immutable `StrategicState` and structural prompt-identity machinery;
8. scaffold the first 20–30-state pilot ledger and bootstrap/measurement checks.

**Scientific invariant:** the consequential action branch, private-report branch, public-report branch, and matched text-only auditor branch must all fork from the same immutable parent state. The activation used for H1 is captured at the final prompt token **before the first generated action token**.

**Scope:** LLM-Deliberation only. CoopEval comes later if the primary result is stable. Avalon and Welfare Diplomacy are gated extensions. **Cattle Trade is not part of this project.**

**Output isolation:** every notebook writes under its own notebook-specific output tree, and every execution gets a separate `RUN_ID` directory. Shared heavyweight assets (the pinned upstream checkout and Hugging Face cache) remain shared.

Official upstream: https://github.com/S-Abdelnabi/LLM-Deliberation


### How to use this notebook

On a fresh Runpod, the intended workflow is simply **Restart Kernel → Run All**.

Defaults are execution-forward:

- dependencies install in the notebook;
- the upstream repository is cloned if absent;
- the default subject checkpoint is `Qwen/Qwen2.5-7B-Instruct` and downloads from Hugging Face if absent;
- the upstream base game is copied into this notebook run's isolated data directory and all agents are pointed at that local HF model;
- the authors' full base-game 24-round simulation path is enabled;
- the subject-model activation smoke test is enabled.

You can override any default with environment variables before executing the notebook. Useful overrides include `SUBJECT_MODEL`, `SUBJECT_REVISION`, `LR_PROJECT_ROOT`, `HF_HOME`, `LR_RUN_NATIVE_SMOKE`, `LR_NATIVE_TEMPERATURE`, and `LR_NATIVE_SMOKE_CMD`.

Output layout for this notebook:

```text
/workspace/latent-reservations/notebook_outputs/00_llm_deliberation_bootstrap/<RUN_ID>/
├── manifests/
├── data/
│   ├── raw/
│   ├── states/
│   ├── activations/
│   ├── derived/
│   ├── frozen/
│   └── generated_games/
└── results/
    ├── tables/
    └── figures/
```

Future notebooks should use the same convention with their own notebook slug, e.g. `01_llm_deliberation_pilot/<RUN_ID>/`. Set `LR_RUN_ID` if you intentionally want a stable run name; otherwise a fresh timestamped run directory is created.

Do **not** paste access tokens into notebook cells. If a selected model later requires authentication, expose `HF_TOKEN` through the Runpod environment/secrets UI.


In [1]:
NOTEBOOK_BUILD = "latent-reservations-notebook00-run-all-v6"
print("=" * 72)
print(f"NOTEBOOK BUILD: {NOTEBOOK_BUILD}")
print("If you do not see this exact line after Restart Kernel -> Run All, you are running an older notebook/tab.")
print("=" * 72)


NOTEBOOK BUILD: latent-reservations-notebook00-run-all-v6
If you do not see this exact line after Restart Kernel -> Run All, you are running an older notebook/tab.


## 0. Bootstrap the Python environment

This cell is intentionally first. The upstream README specifies PyTorch 2.3.0/CUDA 12.1 plus Transformers, Google Cloud AI Platform, OpenAI, and Accelerate. On an empty Runpod we install those dependencies here, plus the analysis packages used by *Latent reservations*.

The exact upstream PyTorch stack is attempted first when PyTorch is missing. If that wheel is unavailable for the notebook's Python version, the cell falls back to the current PyTorch CUDA wheel available through pip and records that fallback.


In [2]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

INSTALL_DEPS = os.environ.get("LR_INSTALL_DEPS", "1") == "1"
bootstrap_install_record = {
    "enabled": INSTALL_DEPS,
    "python": sys.version,
    "commands": [],
    "torch_fallback_used": False,
}


def pip_run(args, *, allow_failure=False):
    cmd = [sys.executable, "-m", "pip", *args]
    print("+", " ".join(cmd))
    p = subprocess.run(cmd, text=True)
    bootstrap_install_record["commands"].append({"cmd": cmd, "returncode": p.returncode})
    if p.returncode != 0 and not allow_failure:
        raise RuntimeError(f"pip command failed with return code {p.returncode}: {' '.join(cmd)}")
    return p.returncode


if INSTALL_DEPS:
    pip_run(["install", "-U", "pip", "setuptools", "wheel"])

    # Match the authors' documented CUDA/PyTorch stack when bootstrapping from empty.
    if importlib.util.find_spec("torch") is None:
        rc = pip_run([
            "install",
            "torch==2.3.0",
            "torchvision==0.18.0",
            "torchaudio==2.3.0",
            "--index-url", "https://download.pytorch.org/whl/cu121",
        ], allow_failure=True)
        if rc != 0:
            bootstrap_install_record["torch_fallback_used"] = True
            print("Pinned PyTorch 2.3/cu121 wheel unavailable; falling back to the current pip CUDA build.")
            pip_run(["install", "torch", "torchvision", "torchaudio"])
    else:
        import torch
        print(f"PyTorch already present: {torch.__version__}; leaving it in place.")

    pip_run([
        "install", "-U",
        "transformers>=4.45,<5",
        "accelerate>=0.30,<2",
        "google-cloud-aiplatform>=1.49,<2",
        "openai>=1,<3",
        "safetensors>=0.4",
        "sentencepiece>=0.2",
        "numpy<2",
        "pandas>=2,<3",
        "scipy>=1.12,<2",
        "scikit-learn>=1.4,<2",
        "matplotlib>=3.8,<4",
        "tqdm>=4.66",
        "pyyaml>=6",
        "huggingface_hub>=0.24,<1",
    ])
else:
    print("Dependency bootstrap disabled with LR_INSTALL_DEPS=0.")

print("Dependency bootstrap complete.")


+ /usr/local/bin/python -m pip install -U pip setuptools wheel
PyTorch already present: 2.8.0+cu128; leaving it in place.
+ /usr/local/bin/python -m pip install -U transformers>=4.45,<5 accelerate>=0.30,<2 google-cloud-aiplatform>=1.49,<2 openai>=1,<3 safetensors>=0.4 sentencepiece>=0.2 numpy<2 pandas>=2,<3 scipy>=1.12,<2 scikit-learn>=1.4,<2 matplotlib>=3.8,<4 tqdm>=4.66 pyyaml>=6 huggingface_hub>=0.24,<1
  Using cached scipy-1.18.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
INFO: pip is looking at multiple versions of scipy to determine which version is compatible with other requirements. This could take a while.
Dependency bootstrap complete.


In [3]:
from __future__ import annotations

import csv
import dataclasses
import hashlib
import json
import os
import platform
import re
import shlex
import shutil
import subprocess
import sys
import time
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Optional

PROJECT_ROOT = Path(os.environ.get("LR_PROJECT_ROOT", "/workspace/latent-reservations")).expanduser().resolve()

# Notebook/run identity. Keep RUN_ID stable if this cell is re-run in the same kernel.
NOTEBOOK_SLUG = "00_llm_deliberation_bootstrap"
RUN_ID = os.environ.get(
    "LR_RUN_ID",
    globals().get("RUN_ID", datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")),
)

# Shared heavyweight assets: reused across notebooks/runs.
VENDOR_DIR = PROJECT_ROOT / "vendor"
HF_CACHE_DIR = Path(
    os.environ.get("HF_HOME", str(PROJECT_ROOT / ".cache" / "huggingface"))
).expanduser().resolve()

# Isolated outputs: every notebook and every execution has its own directory.
NOTEBOOK_OUTPUT_BASE = PROJECT_ROOT / "notebook_outputs" / NOTEBOOK_SLUG
RUN_OUTPUT_DIR = NOTEBOOK_OUTPUT_BASE / RUN_ID
MANIFEST_DIR = RUN_OUTPUT_DIR / "manifests"
DATA_DIR = RUN_OUTPUT_DIR / "data"
RESULTS_DIR = RUN_OUTPUT_DIR / "results"

for p in [
    VENDOR_DIR,
    NOTEBOOK_OUTPUT_BASE,
    RUN_OUTPUT_DIR,
    MANIFEST_DIR,
    DATA_DIR / "raw",
    DATA_DIR / "states",
    DATA_DIR / "activations",
    DATA_DIR / "derived",
    DATA_DIR / "frozen",
    DATA_DIR / "generated_games",
    RESULTS_DIR / "tables",
    RESULTS_DIR / "figures",
    HF_CACHE_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

# Small pointer for downstream notebooks/humans; outputs themselves remain immutable per RUN_ID.
latest_run_pointer = NOTEBOOK_OUTPUT_BASE / "latest_run.json"
latest_run_pointer.write_text(json.dumps({
    "notebook_slug": NOTEBOOK_SLUG,
    "run_id": RUN_ID,
    "run_output_dir": str(RUN_OUTPUT_DIR),
    "updated_at_utc": datetime.now(timezone.utc).isoformat(),
}, indent=2))

os.environ.setdefault("HF_HOME", str(HF_CACHE_DIR))

UPSTREAM_URL = "https://github.com/S-Abdelnabi/LLM-Deliberation.git"
UPSTREAM_DIR = VENDOR_DIR / "llm-deliberation"

# Execution-forward defaults for a fresh GPU Runpod. Set any variable to 0 to disable that action.
RUN_NETWORK_ACTIONS = os.environ.get("LR_RUN_NETWORK_ACTIONS", "1") == "1"
RUN_NATIVE_SMOKE = os.environ.get("LR_RUN_NATIVE_SMOKE", "1") == "1"
RUN_MODEL_SMOKE = os.environ.get("LR_RUN_MODEL_SMOKE", "1") == "1"
AUTO_CONFIGURE_UPSTREAM = os.environ.get("LR_AUTO_CONFIGURE_UPSTREAM", "1") == "1"

SUBJECT_MODEL = os.environ.get("SUBJECT_MODEL", "Qwen/Qwen2.5-7B-Instruct").strip()
SUBJECT_REVISION = os.environ.get("SUBJECT_REVISION", "main").strip()
NATIVE_TEMPERATURE = float(os.environ.get("LR_NATIVE_TEMPERATURE", "0.7"))

# Upstream HF generation hardcodes do_sample=True, so Transformers requires temperature > 0.
if RUN_NATIVE_SMOKE and NATIVE_TEMPERATURE <= 0:
    raise ValueError(
        "LR_NATIVE_TEMPERATURE must be > 0 for the upstream Hugging Face runner because "
        "LLM-Deliberation agent.py hardcodes do_sample=True. Use e.g. 0.7 for native verification."
    )

print(f"PROJECT_ROOT           = {PROJECT_ROOT}")
print(f"NOTEBOOK_SLUG          = {NOTEBOOK_SLUG}")
print(f"RUN_ID                 = {RUN_ID}")
print(f"RUN_OUTPUT_DIR         = {RUN_OUTPUT_DIR}")
print(f"MANIFEST_DIR           = {MANIFEST_DIR}")
print(f"DATA_DIR               = {DATA_DIR}")
print(f"RESULTS_DIR            = {RESULTS_DIR}")
print(f"UPSTREAM_DIR           = {UPSTREAM_DIR}")
print(f"HF_CACHE_DIR           = {HF_CACHE_DIR}")
print(f"RUN_NETWORK_ACTIONS    = {RUN_NETWORK_ACTIONS}")
print(f"AUTO_CONFIGURE_UPSTREAM= {AUTO_CONFIGURE_UPSTREAM}")
print(f"RUN_NATIVE_SMOKE       = {RUN_NATIVE_SMOKE}")
print(f"RUN_MODEL_SMOKE        = {RUN_MODEL_SMOKE}")
print(f"SUBJECT_MODEL          = {SUBJECT_MODEL}")
print(f"SUBJECT_REVISION       = {SUBJECT_REVISION}")
print(f"NATIVE_TEMPERATURE     = {NATIVE_TEMPERATURE}")
print(f"latest_run pointer     = {latest_run_pointer}")


PROJECT_ROOT           = /workspace/latent-reservations
NOTEBOOK_SLUG          = 00_llm_deliberation_bootstrap
RUN_ID                 = 20260813T212827Z
RUN_OUTPUT_DIR         = /workspace/latent-reservations/notebook_outputs/00_llm_deliberation_bootstrap/20260813T212827Z
MANIFEST_DIR           = /workspace/latent-reservations/notebook_outputs/00_llm_deliberation_bootstrap/20260813T212827Z/manifests
DATA_DIR               = /workspace/latent-reservations/notebook_outputs/00_llm_deliberation_bootstrap/20260813T212827Z/data
RESULTS_DIR            = /workspace/latent-reservations/notebook_outputs/00_llm_deliberation_bootstrap/20260813T212827Z/results
UPSTREAM_DIR           = /workspace/latent-reservations/vendor/llm-deliberation
HF_CACHE_DIR           = /workspace/.cache/huggingface
RUN_NETWORK_ACTIONS    = True
AUTO_CONFIGURE_UPSTREAM= True
RUN_NATIVE_SMOKE       = True
RUN_MODEL_SMOKE        = True
SUBJECT_MODEL          = Qwen/Qwen2.5-7B-Instruct
SUBJECT_REVISION       = main
NATIVE_TE

## 1. Capture the execution environment

Save this before changing packages. It is the provenance record for the first pilot and makes later failures distinguishable from scientific failures.

In [4]:
def run_capture(cmd: list[str], cwd: Optional[Path] = None, timeout: int = 120) -> dict[str, Any]:
    started = time.time()
    try:
        p = subprocess.run(
            cmd,
            cwd=str(cwd) if cwd else None,
            text=True,
            capture_output=True,
            timeout=timeout,
            check=False,
        )
        return {
            "cmd": cmd,
            "returncode": p.returncode,
            "stdout": p.stdout,
            "stderr": p.stderr,
            "elapsed_s": round(time.time() - started, 3),
        }
    except Exception as e:
        return {
            "cmd": cmd,
            "returncode": None,
            "stdout": "",
            "stderr": repr(e),
            "elapsed_s": round(time.time() - started, 3),
        }

system_record: dict[str, Any] = {
    "captured_at_utc": datetime.now(timezone.utc).isoformat(),
    "hostname": platform.node(),
    "platform": platform.platform(),
    "python": sys.version,
    "python_executable": sys.executable,
}

try:
    import torch
    system_record["torch"] = torch.__version__
    system_record["cuda_available"] = torch.cuda.is_available()
    system_record["torch_cuda"] = torch.version.cuda
    if torch.cuda.is_available():
        system_record["gpu_name"] = torch.cuda.get_device_name(0)
        system_record["gpu_capability"] = torch.cuda.get_device_capability(0)
        system_record["gpu_memory_bytes"] = torch.cuda.get_device_properties(0).total_memory
except Exception as e:
    system_record["torch_import_error"] = repr(e)

system_record["nvidia_smi"] = run_capture([
    "nvidia-smi",
    "--query-gpu=name,memory.total,driver_version",
    "--format=csv,noheader",
])

system_record_path = MANIFEST_DIR / "runpod_environment.json"
system_record_path.write_text(json.dumps(system_record, indent=2))
print(json.dumps(system_record, indent=2))
print(f"\nWrote: {system_record_path}")

{
  "captured_at_utc": "2026-08-13T21:28:27.402512+00:00",
  "hostname": "66ac93e5f2e7",
  "platform": "Linux-6.8.0-136-generic-x86_64-with-glibc2.39",
  "python": "3.12.3 (main, Aug 14 2025, 17:47:21) [GCC 13.3.0]",
  "python_executable": "/usr/local/bin/python",
  "torch": "2.8.0+cu128",
  "cuda_available": true,
  "torch_cuda": "12.8",
  "gpu_name": "NVIDIA L40S",
  "gpu_capability": [
    8,
    9
  ],
  "gpu_memory_bytes": 47665709056,
  "nvidia_smi": {
    "cmd": [
      "nvidia-smi",
      "--query-gpu=name,memory.total,driver_version",
      "--format=csv,noheader"
    ],
    "returncode": 0,
    "stdout": "NVIDIA L40S, 46068 MiB, 580.159.04\n",
    "stderr": "",
    "elapsed_s": 0.027
  }
}

Wrote: /workspace/latent-reservations/notebook_outputs/00_llm_deliberation_bootstrap/20260813T212827Z/manifests/runpod_environment.json


## 2. Pin the official LLM-Deliberation repository

The upstream repository contains the games, score files/thresholds, simulation code, logs, evaluation notebooks, and multiple model backends. The *Latent reservations* code should wrap it rather than silently modifying it.

This cell clones only when `LR_RUN_NETWORK_ACTIONS=1`. If the repository already exists, it is left untouched.

In [5]:
if not UPSTREAM_DIR.exists():
    if RUN_NETWORK_ACTIONS:
        result = run_capture(["git", "clone", UPSTREAM_URL, str(UPSTREAM_DIR)], timeout=600)
        print(result["stdout"])
        if result["returncode"] != 0:
            raise RuntimeError(result["stderr"])
    else:
        print("Upstream repo not present. Set LR_RUN_NETWORK_ACTIONS=1 and re-run this cell to clone it.")
else:
    print(f"Using existing upstream repo: {UPSTREAM_DIR}")

if UPSTREAM_DIR.exists():
    commit = run_capture(["git", "rev-parse", "HEAD"], cwd=UPSTREAM_DIR)["stdout"].strip()
    branch = run_capture(["git", "branch", "--show-current"], cwd=UPSTREAM_DIR)["stdout"].strip()
    dirty = bool(run_capture(["git", "status", "--porcelain"], cwd=UPSTREAM_DIR)["stdout"].strip())
    print({"commit": commit, "branch": branch, "dirty": dirty})


{'commit': '78e11e04ba89b9d888cb006f57bad696584a32f1', 'branch': 'main', 'dirty': False}


### Repository shape check

This is not yet the scientific verification gate; it simply catches the common failure mode of pinning the wrong repository/branch or an incomplete checkout.

In [6]:
EXPECTED_PATHS = [
    "main.py",
    "agent.py",
    "initial_prompts.py",
    "prompt_utils.py",
    "rounds.py",
    "save_utils.py",
    "games_descriptions",
    "evaluation/evaluate_deals.ipynb",
    "evaluation/score_leakage.py",
]

repo_shape = {}
if UPSTREAM_DIR.exists():
    for rel in EXPECTED_PATHS:
        repo_shape[rel] = (UPSTREAM_DIR / rel).exists()
else:
    repo_shape = {rel: False for rel in EXPECTED_PATHS}

for rel, ok in repo_shape.items():
    print(f"{'OK' if ok else 'MISSING':8s} {rel}")

if UPSTREAM_DIR.exists() and not all(repo_shape.values()):
    print("\nInspect missing paths before writing an adapter; upstream layout may have changed.")

OK       main.py
OK       agent.py
OK       initial_prompts.py
OK       prompt_utils.py
OK       rounds.py
OK       save_utils.py
OK       games_descriptions
OK       evaluation/evaluate_deals.ipynb
OK       evaluation/score_leakage.py


### Locate the exact prompt/history/scoring code paths

The project depends on exact information accounting. Before adapting the environment, locate where the upstream code constructs prompts, stores `prompt/full_answer/public_answer`, loads score files, and evaluates deals. This cell only searches source text; it does not modify upstream.

In [7]:
def grep_repo(pattern: str, suffixes=(".py", ".txt"), max_hits: int = 40):
    if not UPSTREAM_DIR.exists():
        print("Upstream repo not present.")
        return []
    rx = re.compile(pattern, flags=re.IGNORECASE)
    hits = []
    for path in UPSTREAM_DIR.rglob("*"):
        if not path.is_file() or path.suffix not in suffixes:
            continue
        try:
            text = path.read_text(errors="replace")
        except Exception:
            continue
        for i, line in enumerate(text.splitlines(), 1):
            if rx.search(line):
                hits.append((str(path.relative_to(UPSTREAM_DIR)), i, line.strip()))
                if len(hits) >= max_hits:
                    return hits
    return hits

SEARCHES = {
    "history_round_record": r"full_answer|public_answer",
    "score_loading": r"scores_files|load_scores|minimum threshold",
    "prompt_construction": r"history_prompt|scratch_pad|unified_instructions|plan_prompt",
}

for label, pattern in SEARCHES.items():
    print(f"\n## {label}")
    for hit in grep_repo(pattern, max_hits=20):
        print(f"{hit[0]}:{hit[1]}  {hit[2]}")


## history_round_record
save_utils.py:6  def process_answer(full_answer):
save_utils.py:7  public_answer = extract_answer(full_answer)
save_utils.py:8  plan = extract_plan(full_answer)
save_utils.py:9  return public_answer, plan
save_utils.py:12  def save_conversation(history, agent_name,full_answer, prompt,round_assign=[],initial=False):
save_utils.py:21  public_answer, plan  = process_answer(full_answer)
save_utils.py:23  history['content']["rounds"].append({'agent':agent_name, 'prompt': prompt, 'full_answer': full_answer, 'public_answer': public_answer})
prompt_utils.py:9  slot_str = f'. You ({slot['agent']}): {slot['public_answer']}'
prompt_utils.py:11  slot_str = f'. {slot['agent']}: {slot['public_answer']}'
evaluation/score_leakage.py:35  public_answers_file = os.path.join(args.exp_dir, 'public_answers.json')
evaluation/score_leakage.py:46  all_global_answers[answers_count] = round['public_answer']
evaluation/score_leakage.py:64  def get_judge_response(leakage_prompt, public_ans

### Create the project-owned local-HF game configuration

The upstream checkout remains pinned and unmodified. This cell copies `games_descriptions/base` to a run-specific project directory and rewrites only the fifth `config.txt` field (model backend) to `hf_<SUBJECT_MODEL>` for every agent.

Agent count comes from `config.txt`. Issue count is inferred from the first score file as `non-empty score lines - 1`, because the upstream format stores the minimum threshold on the final line. The native round count follows the authors' `4 × agents` convention.


In [8]:
WORK_GAME_DIR = None
AGENTS_NUM = ISSUES_NUM = ROUNDS_NUM = WINDOW_SIZE = None
native_smoke_cmd = os.environ.get("LR_NATIVE_SMOKE_CMD", "").strip()

if AUTO_CONFIGURE_UPSTREAM:
    if not UPSTREAM_DIR.exists():
        raise RuntimeError("Upstream repository is missing; the clone step must succeed before auto-configuration.")

    source_game_dir = UPSTREAM_DIR / "games_descriptions" / "base"
    source_config = source_game_dir / "config.txt"
    if not source_config.exists():
        raise FileNotFoundError(source_config)

    upstream_commit = run_capture(["git", "rev-parse", "HEAD"], cwd=UPSTREAM_DIR)["stdout"].strip()
    safe_model = re.sub(r"[^A-Za-z0-9._-]+", "_", SUBJECT_MODEL)
    WORK_GAME_DIR = DATA_DIR / "generated_games" / f"base_{upstream_commit[:12]}_{safe_model}"

    if not WORK_GAME_DIR.exists():
        shutil.copytree(source_game_dir, WORK_GAME_DIR)
        print(f"Copied upstream base game -> {WORK_GAME_DIR}")
    else:
        print(f"Using existing project-owned game copy: {WORK_GAME_DIR}")

    config_path = WORK_GAME_DIR / "config.txt"
    raw_lines = [line for line in source_config.read_text().splitlines() if line.strip()]
    patched_lines = []
    for line in raw_lines:
        parts = [part.strip() for part in line.split(",")]
        if len(parts) != 5:
            raise ValueError(f"Unexpected upstream config line: {line!r}")
        parts[4] = f"hf_{SUBJECT_MODEL}"
        patched_lines.append(",".join(parts))
    # IMPORTANT: upstream load_setup() does not strip whitespace from filename, role,
    # or incentive fields. Preserve its canonical comma-without-spaces format.
    config_path.write_text("\n".join(patched_lines) + "\n")

    # Validate every generated config reference before launching the expensive native run.
    for patched_line in patched_lines:
        agent_name, file_name, role, incentive, model_name = patched_line.split(",")
        score_ref = WORK_GAME_DIR / "scores_files" / f"{file_name}.txt"
        instruction_ref = WORK_GAME_DIR / "individual_instructions" / incentive / f"{file_name}.txt"
        if not score_ref.exists():
            raise FileNotFoundError(f"Generated config references missing score file: {score_ref}")
        if not instruction_ref.exists():
            raise FileNotFoundError(f"Generated config references missing instruction file: {instruction_ref}")
        if not model_name.startswith("hf_"):
            raise ValueError(f"Generated config has non-HF model backend: {patched_line}")

    AGENTS_NUM = len(patched_lines)
    first_file_name = [part.strip() for part in raw_lines[0].split(",")][1]
    score_file = WORK_GAME_DIR / "scores_files" / f"{first_file_name}.txt"
    score_lines = [line for line in score_file.read_text().splitlines() if line.strip()]
    if len(score_lines) < 2:
        raise ValueError(f"Could not infer issue count from {score_file}")
    ISSUES_NUM = len(score_lines) - 1
    ROUNDS_NUM = 4 * AGENTS_NUM
    WINDOW_SIZE = AGENTS_NUM

    if not native_smoke_cmd:
        native_smoke_args = [
            sys.executable,
            "main.py",
            "--exp_name", f"latent_reservations_smoke_{RUN_ID}",
            "--agents_num", str(AGENTS_NUM),
            "--issues_num", str(ISSUES_NUM),
            "--window_size", str(WINDOW_SIZE),
            "--game_dir", str(WORK_GAME_DIR),
            "--rounds_num", str(ROUNDS_NUM),
            "--temp", str(NATIVE_TEMPERATURE),
            "--hf_home", str(HF_CACHE_DIR),
        ]
        native_smoke_cmd = shlex.join(native_smoke_args)

    print("Configured local HF game:")
    print(config_path.read_text())
    print({
        "agents_num": AGENTS_NUM,
        "issues_num": ISSUES_NUM,
        "rounds_num": ROUNDS_NUM,
        "window_size": WINDOW_SIZE,
        "game_dir": str(WORK_GAME_DIR),
        "temperature": NATIVE_TEMPERATURE,
    })
    print("Native command:")
    print(native_smoke_cmd)
else:
    print("Automatic upstream configuration disabled with LR_AUTO_CONFIGURE_UPSTREAM=0.")


Copied upstream base game -> /workspace/latent-reservations/notebook_outputs/00_llm_deliberation_bootstrap/20260813T212827Z/data/generated_games/base_78e11e04ba89_Qwen_Qwen2.5-7B-Instruct
Configured local HF game:
Mayor,mayor,player,cooperative,hf_Qwen/Qwen2.5-7B-Instruct
Other cities,other_cities,player,cooperative,hf_Qwen/Qwen2.5-7B-Instruct
Local Labour Union,union,player,cooperative,hf_Qwen/Qwen2.5-7B-Instruct
SportCo,SportCo,p1,greedy,hf_Qwen/Qwen2.5-7B-Instruct
Department of Tourism,DoT,p2,greedy,hf_Qwen/Qwen2.5-7B-Instruct
Environmental League,enviroment,player,greedy,hf_Qwen/Qwen2.5-7B-Instruct

{'agents_num': 6, 'issues_num': 5, 'rounds_num': 24, 'window_size': 6, 'game_dir': '/workspace/latent-reservations/notebook_outputs/00_llm_deliberation_bootstrap/20260813T212827Z/data/generated_games/base_78e11e04ba89_Qwen_Qwen2.5-7B-Instruct', 'temperature': 0.7}
Native command:
/usr/local/bin/python main.py --exp_name latent_reservations_smoke_20260813T212827Z --agents_num 6 --issues_

## 3. Write the upstream verification manifest

The file is written as JSON syntax inside a `.yaml` file; JSON is valid YAML, which avoids making PyYAML a bootstrap dependency.

`native_smoke_passed` remains false until an actual upstream simulation succeeds with the chosen local/API backend.

In [9]:
def git_value(args: list[str]) -> str:
    if not UPSTREAM_DIR.exists():
        return ""
    return run_capture(["git", *args], cwd=UPSTREAM_DIR)["stdout"].strip()

license_path = UPSTREAM_DIR / "LICENSE"
license_sha256 = ""
if license_path.exists():
    license_sha256 = hashlib.sha256(license_path.read_bytes()).hexdigest()

manifest = {
    "llm_deliberation": {
        "notebook_slug": NOTEBOOK_SLUG,
        "run_id": RUN_ID,
        "run_output_dir": str(RUN_OUTPUT_DIR),
        "url": UPSTREAM_URL,
        "commit": git_value(["rev-parse", "HEAD"]),
        "branch": git_value(["branch", "--show-current"]),
        "verified_at_utc": datetime.now(timezone.utc).isoformat(),
        "license_file": str(license_path) if license_path.exists() else "",
        "license_sha256": license_sha256,
        "python": platform.python_version(),
        "torch": system_record.get("torch", ""),
        "torch_cuda": system_record.get("torch_cuda", ""),
        "subject_model": SUBJECT_MODEL,
        "subject_revision": SUBJECT_REVISION,
        "hf_cache_dir": str(HF_CACHE_DIR),
        "auto_configured_game_dir": str(WORK_GAME_DIR) if WORK_GAME_DIR else "",
        "agents_num": AGENTS_NUM,
        "issues_num": ISSUES_NUM,
        "rounds_num": ROUNDS_NUM,
        "native_temperature": NATIVE_TEMPERATURE,
        "native_smoke_command": native_smoke_cmd,
        "native_smoke_passed": False,
        "native_smoke_record": "",
        "repo_shape": repo_shape,
        "bootstrap_install_record": bootstrap_install_record,
        "notes": [
            "Upstream checkout is not modified; local model config lives in the project-owned generated game copy.",
            "Do not mark scientifically verified from checkout/import alone.",
            "Scientific gate requires state -> exact prompt -> generation -> legal parsed action -> score -> immutable state -> replay.",
        ],
    }
}

manifest_path = MANIFEST_DIR / "upstream_repositories.yaml"
manifest_path.write_text(json.dumps(manifest, indent=2))
print(manifest_path.read_text())
print(f"Wrote: {manifest_path}")


{
  "llm_deliberation": {
    "notebook_slug": "00_llm_deliberation_bootstrap",
    "run_id": "20260813T212827Z",
    "run_output_dir": "/workspace/latent-reservations/notebook_outputs/00_llm_deliberation_bootstrap/20260813T212827Z",
    "url": "https://github.com/S-Abdelnabi/LLM-Deliberation.git",
    "commit": "78e11e04ba89b9d888cb006f57bad696584a32f1",
    "branch": "main",
    "verified_at_utc": "2026-08-13T21:28:31.892263+00:00",
    "license_file": "/workspace/latent-reservations/vendor/llm-deliberation/LICENSE",
    "license_sha256": "3d640399532779b3f67b2641d079869b094408a452db6fb72868a5abe317458b",
    "python": "3.12.3",
    "torch": "2.8.0+cu128",
    "torch_cuda": "12.8",
    "subject_model": "Qwen/Qwen2.5-7B-Instruct",
    "subject_revision": "main",
    "hf_cache_dir": "/workspace/.cache/huggingface",
    "auto_configured_game_dir": "/workspace/latent-reservations/notebook_outputs/00_llm_deliberation_bootstrap/20260813T212827Z/data/generated_games/base_78e11e04ba89_Qwen_Q

## 4. Run the native upstream simulation

This is **enabled by default**. With the project-owned game copy created above, the notebook launches the authors' native base-game simulation using the selected local Hugging Face model for all agents.

By default this is the full base setup: the agent and issue counts are inferred from upstream, and the round count is `4 × agents` (24 rounds for the six-agent base game). The native HF path uses `do_sample=True`, so this notebook passes `--temp 0.7` by default; override with `LR_NATIVE_TEMPERATURE`, but keep it strictly positive. Set `LR_RUN_NATIVE_SMOKE=0` only when you intentionally want to skip it. `LR_NATIVE_SMOKE_CMD` remains available as a complete command override.


In [10]:
smoke_record = None

def run_stream_capture(cmd: list[str], cwd: Path, timeout: int = 21600) -> dict[str, Any]:
    """Run the native process while streaming merged stdout/stderr live to the notebook."""
    started = time.time()
    print(f"+ {shlex.join(cmd)}")
    print(f"cwd: {cwd}")
    lines: list[str] = []
    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    try:
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="", flush=True)
            lines.append(line)
        returncode = proc.wait(timeout=timeout)
    except Exception:
        proc.kill()
        proc.wait()
        raise
    combined = "".join(lines)
    return {
        "cmd": cmd,
        "returncode": returncode,
        "stdout": combined,
        "stderr": "",  # merged into stdout for live visibility
        "elapsed_s": round(time.time() - started, 3),
    }

if RUN_NATIVE_SMOKE:
    if not UPSTREAM_DIR.exists():
        raise RuntimeError("Clone the upstream repository first.")
    if not native_smoke_cmd:
        raise RuntimeError("No native smoke command is available. Enable auto-configuration or set LR_NATIVE_SMOKE_CMD.")

    print(f"Running in {UPSTREAM_DIR}:\n  {native_smoke_cmd}")
    smoke_record = run_stream_capture(shlex.split(native_smoke_cmd), cwd=UPSTREAM_DIR, timeout=21600)
    smoke_path = MANIFEST_DIR / f"llm_deliberation_native_smoke_{RUN_ID}.json"
    smoke_path.write_text(json.dumps(smoke_record, indent=2))

    manifest = json.loads(manifest_path.read_text())
    manifest["llm_deliberation"]["native_smoke_passed"] = smoke_record["returncode"] == 0
    manifest["llm_deliberation"]["native_smoke_record"] = str(smoke_path)
    manifest_path.write_text(json.dumps(manifest, indent=2))
    print(f"\nUpdated: {manifest_path}")

    if smoke_record["returncode"] != 0:
        output_text = smoke_record.get("stdout", "").strip()
        output_tail = output_text[-16000:] if output_text else "<empty>"
        raise RuntimeError(
            "Native LLM-Deliberation run failed.\n\n"
            f"NOTEBOOK_BUILD: {NOTEBOOK_BUILD}\n\n"
            f"COMMAND:\n{native_smoke_cmd}\n\n"
            f"PROCESS OUTPUT (tail):\n{output_tail}\n\n"
            f"Full subprocess record: {smoke_path}"
        )
else:
    print("Native run skipped because LR_RUN_NATIVE_SMOKE=0.")


Running in /workspace/latent-reservations/vendor/llm-deliberation:
  /usr/local/bin/python main.py --exp_name latent_reservations_smoke_20260813T212827Z --agents_num 6 --issues_num 5 --window_size 6 --game_dir /workspace/latent-reservations/notebook_outputs/00_llm_deliberation_bootstrap/20260813T212827Z/data/generated_games/base_78e11e04ba89_Qwen_Qwen2.5-7B-Instruct --rounds_num 24 --temp 0.7 --hf_home /workspace/.cache/huggingface
+ /usr/local/bin/python main.py --exp_name latent_reservations_smoke_20260813T212827Z --agents_num 6 --issues_num 5 --window_size 6 --game_dir /workspace/latent-reservations/notebook_outputs/00_llm_deliberation_bootstrap/20260813T212827Z/data/generated_games/base_78e11e04ba89_Qwen_Qwen2.5-7B-Instruct --rounds_num 24 --temp 0.7 --hf_home /workspace/.cache/huggingface
cwd: /workspace/latent-reservations/vendor/llm-deliberation

Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.33it/s]
Device set to use cuda:0
=====
SportCo response: <DEAL>A1, B1

## 5. Subject-model + pre-action residual-stream smoke test

This is the critical model-internals check for the project and is **enabled by default**.

The selected checkpoint downloads automatically through Transformers/Hugging Face when absent. The hook records the decoder-block output at the **final prompt token on the initial prompt forward pass** during `generate()`. It captures only once, moves the vector to CPU immediately, and never stores full-sequence hidden states.

Default subject checkpoint: `Qwen/Qwen2.5-7B-Instruct`. Override with `SUBJECT_MODEL` before execution if needed.


In [11]:
def find_decoder_layers(model):
    candidates = [
        ("model.layers", lambda m: getattr(getattr(m, "model", None), "layers", None)),
        ("transformer.h", lambda m: getattr(getattr(m, "transformer", None), "h", None)),
        ("gpt_neox.layers", lambda m: getattr(getattr(m, "gpt_neox", None), "layers", None)),
        ("model.decoder.layers", lambda m: getattr(getattr(getattr(m, "model", None), "decoder", None), "layers", None)),
    ]
    for name, getter in candidates:
        layers = getter(model)
        if layers is not None:
            try:
                if len(layers) > 0:
                    return name, layers
            except TypeError:
                pass
    raise TypeError("Could not locate decoder layers. Add an architecture-specific accessor before proceeding.")

model = tokenizer = None
model_smoke_record = {"enabled": RUN_MODEL_SMOKE, "subject_model": SUBJECT_MODEL}

if RUN_MODEL_SMOKE:
    if not SUBJECT_MODEL:
        raise RuntimeError("Set SUBJECT_MODEL before LR_RUN_MODEL_SMOKE=1.")
    try:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
    except ImportError as e:
        raise ImportError("Install torch + transformers + accelerate in the Runpod environment first.") from e

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available; do not proceed with the main subject-model run.")

    tokenizer = AutoTokenizer.from_pretrained(SUBJECT_MODEL, revision=SUBJECT_REVISION, cache_dir=str(HF_CACHE_DIR))
    model = AutoModelForCausalLM.from_pretrained(
        SUBJECT_MODEL,
        revision=SUBJECT_REVISION,
        torch_dtype=torch.bfloat16,
        device_map={"": 0},
        low_cpu_mem_usage=True,
        cache_dir=str(HF_CACHE_DIR),
    )
    model.eval()

    layer_path, layers = find_decoder_layers(model)
    layer_index = len(layers) // 2
    capture: dict[str, Any] = {"done": False}

    def capture_final_prompt_token(_module, _inputs, output):
        if capture["done"]:
            return
        hidden = output[0] if isinstance(output, tuple) else output
        if getattr(hidden, "ndim", 0) == 3:
            capture["h"] = hidden[:, -1, :].detach().to(dtype=torch.float16, device="cpu").clone()
            capture["seq_len"] = int(hidden.shape[1])
            capture["hidden_size"] = int(hidden.shape[-1])
            capture["done"] = True

    handle = layers[layer_index].register_forward_hook(capture_final_prompt_token)
    prompt = "You are choosing between Action A and Action B. Reply with exactly one: A or B.\nChoice:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=1,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    handle.remove()

    new_tokens = generated[0, inputs["input_ids"].shape[1]:]
    generated_text = tokenizer.decode(new_tokens, skip_special_tokens=True)

    assert capture.get("done"), "Hook never fired."
    assert capture["seq_len"] == inputs["input_ids"].shape[1], (
        "Hook did not capture the initial prompt pass. Inspect this architecture's generation path before continuing."
    )
    assert capture["h"].shape[0] == 1

    model_smoke_record.update({
        "passed": True,
        "layer_container": layer_path,
        "num_layers": len(layers),
        "captured_layer": layer_index,
        "prompt_tokens": int(inputs["input_ids"].shape[1]),
        "captured_shape": list(capture["h"].shape),
        "captured_dtype": str(capture["h"].dtype),
        "generated_first_token_text": generated_text,
    })
    torch.save(capture["h"], DATA_DIR / "activations" / "model_smoke_final_prompt_token.pt")
    print(json.dumps(model_smoke_record, indent=2))
else:
    model_smoke_record["passed"] = False
    model_smoke_record["reason"] = "skipped"
    print("Model smoke skipped because LR_RUN_MODEL_SMOKE=0.")

(MANIFEST_DIR / "subject_model_smoke.json").write_text(json.dumps(model_smoke_record, indent=2))

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{
  "enabled": true,
  "subject_model": "Qwen/Qwen2.5-7B-Instruct",
  "passed": true,
  "layer_container": "model.layers",
  "num_layers": 28,
  "captured_layer": 14,
  "prompt_tokens": 21,
  "captured_shape": [
    1,
    3584
  ],
  "captured_dtype": "torch.float16",
  "generated_first_token_text": " A"
}


308

## 6. Immutable frozen-state schema

`ground_truth` is intentionally stored separately from text rendered to the subject/auditor. It must not be interpolated into a prompt automatically.

The parent snapshot hash is the identity anchor across all branches.

In [12]:
@dataclass(frozen=True)
class StrategicState:
    state_id: str
    source_id: str
    environment: str

    public_text: str
    private_text: str
    exact_subject_text: str

    legal_actions: tuple[str, ...]
    ground_truth: dict[str, Any]
    metadata: dict[str, Any]

    upstream_repo: str
    upstream_commit: str
    adapter_version: str


def canonical_json(obj: Any) -> str:
    return json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(",", ":"), default=str)


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def parent_snapshot_hash(state: StrategicState) -> str:
    return sha256_text(canonical_json(asdict(state)))


def save_state(state: StrategicState, directory: Path = DATA_DIR / "states") -> Path:
    directory.mkdir(parents=True, exist_ok=True)
    payload = asdict(state)
    payload["legal_actions"] = list(state.legal_actions)
    payload["parent_snapshot_hash"] = parent_snapshot_hash(state)
    path = directory / f"{state.state_id}.json"
    tmp = path.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(payload, indent=2, ensure_ascii=False))
    tmp.replace(path)
    return path


def load_state(path: Path) -> StrategicState:
    payload = json.loads(path.read_text())
    expected_hash = payload.pop("parent_snapshot_hash")
    payload["legal_actions"] = tuple(payload["legal_actions"])
    state = StrategicState(**payload)
    actual_hash = parent_snapshot_hash(state)
    if actual_hash != expected_hash:
        raise ValueError(f"State hash mismatch: {actual_hash} != {expected_hash}")
    return state

print("StrategicState schema ready.")

StrategicState schema ready.


## 7. Structural information matching for H1

The auditor is allowed to receive a different **task instruction** (predict the subject rather than act as the subject), but its decision-relevant observation must be identical to the subject's observation.

The safest implementation is structural: both renderers consume the same immutable observation string and we hash that shared component separately from branch instructions.

In [13]:
@dataclass(frozen=True)
class ObservationBundle:
    state_id: str
    observation_text: str
    observation_hash: str


def observation_bundle(state: StrategicState) -> ObservationBundle:
    # exact_subject_text must already contain every public + private textual fact visible at decision time.
    text = state.exact_subject_text
    return ObservationBundle(
        state_id=state.state_id,
        observation_text=text,
        observation_hash=sha256_text(text),
    )


def render_action_prompt(state: StrategicState) -> tuple[str, ObservationBundle]:
    obs = observation_bundle(state)
    instruction = "\n\n[DECISION]\nChoose the consequential action now. Follow the required output format."
    return obs.observation_text + instruction, obs


def render_private_report_prompt(state: StrategicState) -> tuple[str, ObservationBundle]:
    obs = observation_bundle(state)
    instruction = "\n\n[PRIVATE REPORT]\nPrivately report your current preference/value assessment in the required format."
    return obs.observation_text + instruction, obs


def render_auditor_prompt(state: StrategicState) -> tuple[str, ObservationBundle]:
    obs = observation_bundle(state)
    instruction = (
        "\n\n[AUDITOR TASK]\nPredict the consequential action the subject model will choose. "
        "You receive the same decision-relevant textual observation as the subject and no activations or future outputs."
    )
    return obs.observation_text + instruction, obs


def assert_matched_information(state: StrategicState) -> None:
    _, action_obs = render_action_prompt(state)
    _, auditor_obs = render_auditor_prompt(state)
    assert action_obs.observation_hash == auditor_obs.observation_hash
    assert action_obs.observation_text == auditor_obs.observation_text

print("Matched-information assertion ready.")

Matched-information assertion ready.


## 8. First frozen-state acceptance test

Once the LLM-Deliberation adapter can extract a real decision state, replace the placeholder below with that adapter output. The acceptance test should pass before generating a 20–30-state pilot.

A valid state must support:

```text
source state
 -> exact subject prompt
 -> pre-action activation
 -> legal action
 -> parsed behavioral label / score
 -> immutable serialized state
 -> replay from the same state
```

In [14]:
def validate_state_static(state: StrategicState) -> list[str]:
    problems = []
    if state.environment != "llm_deliberation":
        problems.append("environment must be llm_deliberation in Notebook 00")
    if not state.state_id or not state.source_id:
        problems.append("state_id/source_id missing")
    if not state.exact_subject_text.strip():
        problems.append("exact_subject_text is empty")
    if not state.legal_actions:
        problems.append("no legal actions")
    if not state.upstream_commit:
        problems.append("upstream commit missing")
    if state.ground_truth and canonical_json(state.ground_truth) in state.exact_subject_text:
        problems.append("ground_truth appears serialized verbatim into subject text")
    try:
        assert_matched_information(state)
    except Exception as e:
        problems.append(f"auditor information mismatch: {e}")
    return problems

# Deliberately no fake scientific state is created here.
# The first real StrategicState must come from the verified LLM-Deliberation adapter.
print("Static state validator ready; waiting for first real adapter state.")

Static state validator ready; waiting for first real adapter state.


## 9. Pilot ledger for 20–30 frozen states

The ledger records provenance and branch identity before any probe is fit. It is intentionally wider than the MVP so later branches do not require changing the data contract.

In [15]:
PILOT_COLUMNS = [
    "slot_id",
    "state_id",
    "source_id",
    "split_group",
    "parent_snapshot_hash",
    "branch_id",
    "branch",
    "condition_id",
    "subject_observation_hash",
    "subject_prompt_hash",
    "auditor_observation_hash",
    "action_raw",
    "action_parsed",
    "behavior_label",
    "behavior_score",
    "private_report_raw",
    "private_report_value",
    "public_report_raw",
    "public_report_value",
    "auditor_prediction",
    "activation_path",
    "activation_layers",
    "valid",
    "error_type",
    "notes",
]

pilot_ledger_path = DATA_DIR / "derived" / "pilot_ledger.csv"
if not pilot_ledger_path.exists():
    with pilot_ledger_path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=PILOT_COLUMNS)
        writer.writeheader()
        for i in range(30):
            row = {c: "" for c in PILOT_COLUMNS}
            row["slot_id"] = i
            writer.writerow(row)
    print(f"Created 30-slot pilot ledger: {pilot_ledger_path}")
else:
    print(f"Pilot ledger already exists; not overwriting: {pilot_ledger_path}")

Created 30-slot pilot ledger: /workspace/latent-reservations/notebook_outputs/00_llm_deliberation_bootstrap/20260813T212827Z/data/derived/pilot_ledger.csv


## 10. Split-purity guard

All branches from one source game/trajectory/state must stay in the same split. The first notebook does not fit probes, but it defines the invariant now so accidental leakage is caught before scaling.

In [16]:
def grouped_split(groups: list[str], seed: int = 1729, train=0.60, val=0.20):
    import random

    unique = sorted(set(groups))
    rng = random.Random(seed)
    rng.shuffle(unique)
    n = len(unique)
    n_train = int(round(n * train))
    n_val = int(round(n * val))

    split_of = {}
    for g in unique[:n_train]:
        split_of[g] = "train"
    for g in unique[n_train:n_train + n_val]:
        split_of[g] = "validation"
    for g in unique[n_train + n_val:]:
        split_of[g] = "test"
    return split_of


def assert_branch_split_purity(rows: list[dict[str, Any]], split_of: dict[str, str]) -> None:
    seen = {}
    for row in rows:
        group = row["split_group"]
        state = row["state_id"]
        split = split_of[group]
        if state in seen and seen[state] != split:
            raise AssertionError(f"State {state} appears in multiple splits")
        seen[state] = split

print("Grouped split guard ready. Do not split individual branches independently.")

Grouped split guard ready. Do not split individual branches independently.


## 11. Bootstrap and measurement gate status

Do not move to broader environments merely because the notebook executes. The project advances only when the measurement chain is valid.

In [17]:
def bool_file(path: Path) -> bool:
    return path.exists() and path.stat().st_size > 0

manifest_now = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
upstream_entry = manifest_now.get("llm_deliberation", {})

measurement_gate = {
    "output_isolated_by_notebook": RUN_OUTPUT_DIR.parent == NOTEBOOK_OUTPUT_BASE,
    "run_output_dir": str(RUN_OUTPUT_DIR),
    "environment_recorded": bool_file(MANIFEST_DIR / "runpod_environment.json"),
    "upstream_repo_present": UPSTREAM_DIR.exists(),
    "upstream_commit_recorded": bool(upstream_entry.get("commit")),
    "native_smoke_passed": bool(upstream_entry.get("native_smoke_passed", False)),
    "subject_model_smoke_passed": bool(model_smoke_record.get("passed", False)),
    "immutable_state_schema_ready": True,
    "matched_information_assertion_ready": True,
    "pilot_ledger_ready": pilot_ledger_path.exists(),
    "first_real_state_validated": False,
    "20_30_state_pilot_complete": False,
    "h1_pilot_estimate_complete": False,
}

measurement_gate_path = RESULTS_DIR / "tables" / "measurement_gate.json"
measurement_gate_path.write_text(json.dumps(measurement_gate, indent=2))
print(json.dumps(measurement_gate, indent=2))
print(f"\nWrote: {measurement_gate_path}")

{
  "output_isolated_by_notebook": true,
  "run_output_dir": "/workspace/latent-reservations/notebook_outputs/00_llm_deliberation_bootstrap/20260813T212827Z",
  "environment_recorded": true,
  "upstream_repo_present": true,
  "upstream_commit_recorded": true,
  "native_smoke_passed": true,
  "subject_model_smoke_passed": true,
  "immutable_state_schema_ready": true,
  "matched_information_assertion_ready": true,
  "pilot_ledger_ready": true,
  "first_real_state_validated": false,
  "20_30_state_pilot_complete": false,
  "h1_pilot_estimate_complete": false
}

Wrote: /workspace/latent-reservations/notebook_outputs/00_llm_deliberation_bootstrap/20260813T212827Z/results/tables/measurement_gate.json


## Exit condition for Notebook 00

A normal fresh-Runpod **Run All** should now automatically satisfy the infrastructure half of this gate:

- dependencies installed in the active kernel environment;
- the LLM-Deliberation repository pinned to an exact commit;
- a project-owned base-game config generated for the local subject checkpoint;
- the author-native 24-round simulation path completed successfully;
- the selected subject checkpoint downloaded and loaded on the L40S;
- the residual hook captured the final prompt-token vector on the initial generation pass.

The remaining scientific gate is intentionally not faked in Notebook 00:

- one **real** LLM-Deliberation decision state serialized and replayed identically;
- action and auditor branches share the same observation hash;
- the action is legal and parsable, and its score/behavioral label is reproducible;
- no future action/output or hidden ground-truth field enters the matched auditor prompt.

Then build **Notebook 01: 20–30-state pilot + all-layer probe**, not another environment integration.
